<div style="display:fill; background-color:#000000;border-radius:5px;">
    <p style="font-size:300%; color:white;text-align:center";>Abstract</p>
</div>

This notebook contains my model to predict the surival or the death of the Titanic passengers. This work is the continuation of my EDA of the Titanic dataset that can be found [here](https://www.kaggle.com/arnrob/titanic-eda).

Based on the findings of the EDA, I implemented the following approach :  
1/ Create a generic model that can be applied to all passengers.   
2/ Create a specific model that can be applied to passengers travelling in groups (e.g. family or groups of friends...)  

For the generic model, I got my best result (0.78229) using a RandomForestClassifier with this set of features :   "Embarked", "Title", "Sex", "FareCat", "Pclass", "AgeCat", "FamilyType", "GroupSize", "CabinGroup".

For the specific model that can be applied to passengers travelling in group, I got my best result using a SVC with this set of features :  
"Embarked", "Title", "Sex", "FareCat", "Pclass", "AgeCat", "FamilyType", "GroupSize", "CabinGroup", "MixedGroup", "NbSurvivors", "NbDead", "NbMaleSurvivors", "NbFemaleDead".

**Overwriting the predictions of the generic model with the ones of the specific model for passengers travelling in group boosted the performance of the model from 0.78229 to 0.80143 on the test set.**

I'm sure that there is plenty of room for improvement. I'll come back to this competition one day and explore new strategies. 

<div style="display:fill; background-color:#000000;border-radius:5px;">
    <p style="font-size:300%; color:white;text-align:center";>Agenda</p>
</div>

Detailed agenda of this notebook : <a id='Agenda'></a> 

- Data preparation : 

    - [Loading libraries and datasets](#Loading-data)
    - [Data cleaning](#Data-cleaning)
    - [Handling missing values](#Handling-NAs)
    - [Checking the new dataset](#Checking-data-1)


- Feature Engineering :

    - [Feature engineering (all passengers)](#Feature-engineering-1)
    - [Feature engineering (passengers in group)](#Feature-engineering-2)
    - [Checking the new dataset](#Checking-data-2)


- Generic Model :

    - [Splitting train and test sets](#Splitting-1)
    - [Encoding categorical features and scaling numeric features](#Encoding-1)
    - [Random Forest Classifier - Hyperparmater Tuning](#RFC-Tuning)
    - [Random Forest Classifier - Prediction](#RFC-Tuning)


- Specific Model :

    - [Splitting train and test sets](#Splitting-2)
    - [Encoding categorical features and scaling numeric features](#Encoding-2)
    - [Support Vector Classifier - Hyperparmater Tuning](#SVC-Tuning)
    - [Support Vector Classifier - Predictions](#SVC-Predictions)


- File Submission :

    - [Merging the predictions of the 2 models and submit the predictions](#Merging)

# Loading libraries and datasets <a id='Loading-data'></a>
[Go back to the agenda](#Agenda)

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler 
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

train = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")
full = pd.concat([train, test])

seed = 42

# Data cleaning <a id='Data-cleaning'></a>
[Go back to the agenda](#Agenda)

In [2]:
###############################
# Extracting Title and LastName
###############################
full["Title"] = full["Name"].apply(lambda x: str.strip(str.split(str.split(x,",")[1],".")[0] + "."))
full["LastName"] = full["Name"].apply(lambda x: str.strip(str.split(x,",")[0]))

#################
# Grouping titles
#################
def transformTitle(title):
    if(title in ['Mr.','Mrs.','Miss.','Master.']):
        return title
    elif (title in  ['Ms.', 'Mme.']):
        return 'Mrs.'
    elif(title=='Mlle.'):
        return 'Miss.'
    elif(title in  ['Sir', 'Don.', 'Jonkheer.', 'Dr.', 'Col.', 'Capt.', 'Major.', 'Rev.']): 
        return 'Mr.'
    elif(title in ['Lady.', 'the Countess.']):
        return 'Mrs.'
    else:
        return 'Mr.'
    
full['Title'] = full['Title'].apply(lambda x: transformTitle(x))

# Fixing some discrepancies
# The above change created a discrepancy. Dr Alice Leader is a woman. Therefore, let's correct it with a Mrs.
full.loc[796,'Title']='Mrs.'
# Changing this 11 year old passenger title from Mr. to Master?
full.loc[731,'Title']='Master.'

# Handling missing values <a id='Handling-NAs'></a>
[Go back to the agenda](#Agenda)

In [3]:
################################
# Handling missing values -- Age
################################
def estimate_age(age, title, sex, pclass, parch):
    if(np.isnan(age)):
        if(title == 'Mr.'):
            if(pclass==1):
                return 42
            elif(pclass==2):
                return 33
            else:
                return 28
        elif(title == 'Mrs.'):
            if(pclass==1):
                return 40
            elif(pclass==2):
                return 33
            else:
                return 33
        elif(title == 'Master.'):
            if(pclass==1):
                return 5
            elif(pclass==2):
                return 3
            else:
                return 5
        elif(title == 'Miss.' and parch==0):
            if(pclass==1):
                return 34
            elif(pclass==2):
                return 30
            else:
                return 21
        elif(title == 'Miss.' and parch > 0):
            if(pclass==1):
                return 21
            elif(pclass==2):
                return 10
            else:
                return 6
    return age

full["Age"] = full.apply(lambda x: estimate_age(x["Age"],x["Title"],x["Sex"],x["Pclass"],x["Parch"]), axis=1)

#####################################
# Handling missing values -- Embarked
#####################################
full['Embarked'] = full['Embarked'].fillna('S')

#################################
# Handling missing values -- Fare
#################################
full['Fare'] = full['Fare'].fillna(8)

##################################
# Handling missing values -- Cabin
##################################
full['Cabin'] = full['Cabin'].fillna('M')
full.loc[full[full['Cabin']=='T'].index,'Cabin'] = 'M' #there is only one record with Cabin=T...

# Checking the new dataset <a id='Checking-data-1'></a>
[Go back to the agenda](#Agenda)

In [4]:
full.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,LastName
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,M,S,Mr.,Braund
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs.,Cumings
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,M,S,Miss.,Heikkinen
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Mrs.,Futrelle
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,M,S,Mr.,Allen


<div style="display:fill; background-color:#000000;border-radius:5px;">
    <p style="font-size:300%; color:white;text-align:center";>Feature Engineering</p>
</div>

# Feature Engineering (all passengers) <a id='Feature-engineering-1'></a>
[Go back to the agenda](#Agenda)

The code below creates these new features:
- "Title" will be extracted from "Name" and we will group title into 4 categories "Mr.", "Mrs.", "Master.", "Miss."
- "AgeCat" will be created from "Age" by grouping close ages into the same category
- "FareCat" will be created from "Fare" by grouping close fares into the same category
- "CabinGroup" will be created from "Cabin" by grouping cabins as follows : group 'A', group 'BDEF', group 'C' and group 'GM'
- "FamilySize" will be created from "Parch" and "SibSp" (FamilySize=Parch+SibSp+1) 
- "FamilyType" will be created from "FamilySize". FamilyType will be set to either alone or small (2-4 members) or big
- "GroupSize" will be created from "Ticket" (GroupSize=the number of passengers having the same ticket)
- "GroupType" will be created from "GroupSize". GroupType will be set to either alone or small (2-4 members) or big

In [5]:
full['AgeCat'] = pd.cut(full['Age'], 16, right=False, labels=range(16))
full['FareCat'] = pd.cut(full['Fare'], 25, right=False, labels=range(25))
full['FamilySize'] = full['SibSp'] + full['Parch'] + 1
full['FamilyType'] = full['FamilySize'].apply(lambda x: 'alone' if x==1 else 'small' if x in [2,3,4] else 'big')
full['GroupSize'] = full['Ticket'].map(full.groupby('Ticket')['PassengerId'].count())
full['GroupType'] = full['GroupSize'].apply(lambda x: 'alone' if x==1 else 'small' if x in [2,3,4] else 'big')
# Adding CabinGroup
full['CabinLetter'] = full['Cabin'].apply(lambda x: x[0])
full['CabinGroup'] = full['CabinLetter']
di = {'A':'A', 'B':'BDEF', 'C':'C', 'D':'BDEF', 'E':'BDEF', 'F':'BDEF', 'G':'GM', 'M':'GM'}
full.replace({'CabinGroup': di}, inplace=True)

# Feature Engineering (passengers in group) <a id='Feature-engineering-2'></a>
[Go back to the agenda](#Agenda)

The code below creates these new features that apply to passengers travelling in group:  
- "MixedGroup" means the Group is mixed i.e. composed of male and female.
- "NbSurvivors" = nb of survivors in the train set for this group 
- "NbDead" = nb of dead in the train set for this group 
- "NbMaleSurvivors" = nb of males who survived in the train set for this group 
- "NbFemaleSurvivors" = nb of females who survived in the train set for this group 
- "NbMaleDead" = nb of males who perished in the train set for this group 
- "FemaleDead" = nb of females who perished in the train set for this group 

In [6]:
# Feature Engineering for Families & Groups
# MixedGroup=1 means the Group is mixed i.e. composed of male and female.
full['Sex_tmp'] = full['Sex'].apply(lambda x: 1 if x=='male' else 0) 
full['Sex_mean_tmp'] = full['Ticket'].map(full.groupby('Ticket')['Sex_tmp'].mean())
full['MixedGroup'] = full['Sex_mean_tmp'].apply(lambda x: 0 if round(x)==x else 1)
full.drop(['Sex_tmp','Sex_mean_tmp'], axis=1, inplace=True)
# 'NbSurvivors' = minimum number of survivors in the passenger's Group and # 'NbDead' = minimum number of dead in the passenger's Group
full['NbSurvivors'] = full['Ticket'].map(full[full['Survived']==1].groupby('Ticket')['Survived'].count()).fillna(0)
full['NbDead'] = full['Ticket'].map(full[full['Survived']==0].groupby('Ticket')['Survived'].count()).fillna(0)
# Let's repeat with male & female split
full['NbMaleSurvivors'] = full['Ticket'].map(full[(full['Survived']==1) & (full['Sex']=='male')].groupby('Ticket')['Survived'].count()).fillna(0)
full['NbMaleDead'] = full['Ticket'].map(full[(full['Survived']==0) & (full['Sex']=='male')].groupby('Ticket')['Survived'].count()).fillna(0)
full['NbFemaleSurvivors'] = full['Ticket'].map(full[(full['Survived']==1) & (full['Sex']=='female')].groupby('Ticket')['Survived'].count()).fillna(0)
full['NbFemaleDead'] = full['Ticket'].map(full[(full['Survived']==0) & (full['Sex']=='female')].groupby('Ticket')['Survived'].count()).fillna(0)

# Checking the new dataset <a id='Checking-data-2'></a>
[Go back to the agenda](#Agenda)

In [7]:
full.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,GroupType,CabinLetter,CabinGroup,MixedGroup,NbSurvivors,NbDead,NbMaleSurvivors,NbMaleDead,NbFemaleSurvivors,NbFemaleDead
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,...,alone,M,GM,0,0.0,1.0,0.0,1.0,0.0,0.0
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,...,small,C,C,1,1.0,0.0,0.0,0.0,1.0,0.0
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,...,alone,M,GM,0,1.0,0.0,0.0,0.0,1.0,0.0
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,...,small,C,C,1,1.0,1.0,0.0,1.0,1.0,0.0
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,...,alone,M,GM,0,0.0,1.0,0.0,1.0,0.0,0.0


<div style="display:fill; background-color:#000000;border-radius:5px;">
    <p style="font-size:300%; color:white;text-align:center";>Generic Model</p>
</div>

The generic model can be applied to all the passengers. To select this model : 
- I benchmarked 3 classifiers : Random Forest Classifier, Logistic Regression and Support Vector Classifier. 
- I used GridSearchCV into a pipeline for tuning the hyperparameters.
- I compared different set of features.

**I achieved my best result (0.78229) against the test set with:**

**- a Random Forest Classifier**   

**- this set of hyperparameters n_estimators=50, min_samples_leaf=5, min_samples_split=4, max_depth=5, max_features='auto'**  

**- this set of features "Embarked", "Title", "Sex", "FareCat", "Pclass", "AgeCat", "FamilyType", "GroupSize", "CabinGroup".**

# Splitting train and test sets <a id='Splitting-1'></a>
[Go back to the agenda](#Agenda)

In [8]:
train_full = full[full['Survived'].isna()==False].copy()
test_full = full[full['Survived'].isna()==True].copy()

y_train_full = train_full['Survived']

X_train_full = train_full[['Embarked','Title','Sex','FareCat','Pclass','AgeCat','FamilyType','GroupSize','CabinGroup']]

X_test_full = test_full[['Embarked','Title','Sex','FareCat','Pclass','AgeCat','FamilyType','GroupSize','CabinGroup']]

#  Encoding categorical features and scaling numeric features <a id='Encoding-1'></a>
[Go back to the agenda](#Agenda)

In [9]:
preprocessor = ColumnTransformer(
    [('onehot', OneHotEncoder(), ['Embarked', 'Title','Sex','Pclass','FamilyType','CabinGroup']), 
     ('scale', StandardScaler(), ['FareCat','AgeCat','GroupSize'])], 
    remainder='passthrough') 

#  Random Forest Classifier - Tuning Hyperparameters<a id='RFC-Tuning'></a>
[Go back to the agenda](#Agenda)

In [10]:
param_grid = {
        #RandomForestClassifier
        'classifier__n_estimators': [100, 50, 200, 300, 600, 1000, 1500, 2000],
        'classifier__min_samples_leaf': [1, 5, 10, 20],
        'classifier__min_samples_split' : [2, 3, 4, 6],
        'classifier__max_depth' : [None, 3, 5, 7],
        'classifier__max_features': ['auto', None],
        'classifier__random_state' : [seed],
        'classifier__oob_score' : ['True'],
        'classifier__n_jobs' : [-1]
    }

#pipe = Pipeline(steps=[('preprocessor', preprocessor),('classifier', RandomForestClassifier())])
#grid = GridSearchCV(pipe, cv=10, n_jobs=-1, param_grid=param_grid, scoring='accuracy', refit=True)
#grid.fit(X_train_full, y_train_full)

################################
## Let's display the best models
################################
#results_df = pd.DataFrame(grid.cv_results_).sort_values(by=['rank_test_score'])
#for p in results_df['params'][0]:
#    colname = str.split(p,'__')[1]
#    results_df[colname] = results_df["params"].apply(lambda x: x[p])
#results_df[results_df['rank_test_score']<10][['n_estimators', 'min_samples_leaf', 'min_samples_split', 'max_depth','max_features','rank_test_score', 'mean_test_score', 'std_test_score']]

#  Random Forest Classifier - Predictions <a id='RFC-Prediction'></a>
[Go back to the agenda](#Agenda)

In [11]:
pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(n_estimators=50, min_samples_leaf=5, min_samples_split=4, max_depth=5, max_features='auto', random_state=seed))]) 

pipe.fit(X_train_full, y_train_full)
y_test_full = pipe.predict(X_test_full)

<div style="display:fill; background-color:#000000;border-radius:5px;">
    <p style="font-size:300%; color:white;text-align:center";>Specific Model</p>
</div>

The specific model can be applied to passengers travelling in group having one or more group members in the train set. In my [EDA](https://www.kaggle.com/arnrob/titanic-eda), I found that :
1. Circa 60% of the families and 85% of the groups shared the same fate (all survived or all perished)
2. The survival of a male very often implied the survival of the entire family
3. The death of a female implied the death of the entire family

I created this model to benefit from these patterns that (may) exist in groups.

For these passengers, I added extra features :
- "MixedGroup" : means the Group is mixed i.e. composed of male and female.
- "NbSurvivors" : nb of survivors in the train set for this group.
- "NbDead" : nb of dead in the train set for this group.
- "NbMaleSurvived" : nb of males who survived in the train set for this group.
- "NbFemaleDied" : nb of females who perished in the train set for this group.

To select this model : 
- I benchmarked 3 classifiers : Random Forest Classifier, Logistic Regression and Support Vector Classifier. 
- I used GridSearchCV into a pipeline for tuning the hyperparameters.

**Overwriting the predictions of the generic model with the ones of the specific model for passengers travelling in group boosted the performance of the model from 0.78229 to 0.80143 on the test set.**

# Splitting train and test sets <a id='Splitting-2'></a>
[Go back to the agenda](#Agenda)

In [12]:
full_group = full[(full['GroupSize']>1) & (full['NbSurvivors'] + full['NbDead']>0)].copy()

# Let's get the train and test set for passengers with group
train_group = full_group[full_group['Survived'].isna()==False].copy()
test_group = full_group[full_group['Survived'].isna()==True].copy()

y_train_group = train_group['Survived']
X_train_group = train_group[['Embarked','Title','Sex','FareCat','Pclass','AgeCat','GroupType','GroupSize','CabinGroup','MixedGroup','NbSurvivors','NbDead','NbMaleSurvivors','NbFemaleDead']]

X_test_group = test_group[['Embarked','Title','Sex','FareCat','Pclass','AgeCat','GroupType','GroupSize','CabinGroup','MixedGroup','NbSurvivors','NbDead','NbMaleSurvivors','NbFemaleDead']]

#  Encoding categorical features and scaling numeric features <a id='Encoding-2'></a>
[Go back to the agenda](#Agenda)

In [13]:
preprocessor = ColumnTransformer(
    #[('onehot', OneHotEncoder(handle_unknown='ignore'), ['Embarked', 'Title','Sex','Pclass','GroupType']), #handle_unknown='ignore' there is a master in X_train_alone
    [('onehot', OneHotEncoder(), ['Embarked', 'Title','Sex','Pclass','GroupType','CabinGroup']), 
     ('scale', StandardScaler(), ['FareCat','AgeCat','GroupSize','NbSurvivors','NbDead','NbMaleSurvivors','NbFemaleDead'])], 
    remainder='passthrough') #passthrough for 'MixedGroup' 

#  SVC - Tuning Hyperparameters <a id='SVC-Tuning'></a>
[Go back to the agenda](#Agenda)

In [14]:
param_grid = {
        #SVC
        'classifier__C': [1, 0.2, 0.4, 0.6, 0.8, 1.2], 
        'classifier__kernel': ['rbf', 'poly'], 
        'classifier__degree': [3, 2, 4], # Used by polynomial kernel function. Ignored by all other kernels.
        'classifier__gamma': ['scale', 'auto'],
        'classifier__random_state' : [seed]
    }

#pipe = Pipeline(steps=[('preprocessor', preprocessor),('classifier', SVC())])
#grid = GridSearchCV(pipe, cv=10, n_jobs=-1, param_grid=param_grid, scoring='accuracy', refit=True)
#grid.fit(X_train_group, y_train_group)

################################
## Let's display the best models
################################
#results_df = pd.DataFrame(grid.cv_results_).sort_values(by=['rank_test_score'])
#for p in results_df['params'][0]:
#    colname = str.split(p,'__')[1]
#    results_df[colname] = results_df["params"].apply(lambda x: x[p])
#results_df[results_df['rank_test_score']<10][['C', 'kernel', 'degree', 'gamma', 'rank_test_score', 'mean_test_score', 'std_test_score']]

#  SVC - Predictions <a id='SVC-Predictions'></a>
[Go back to the agenda](#Agenda)

In [15]:
# C=0.4, kernel='rbf', degree=4, gamma='scale' --> CV result : rank_test_score=1
pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', SVC(C=0.4, kernel='rbf', degree=4, gamma='scale', random_state=seed))]) 

pipe.fit(X_train_group, y_train_group)
y_test_group = pipe.predict(X_test_group)

<div style="display:fill; background-color:#000000;border-radius:5px;">
    <p style="font-size:300%; color:white;text-align:center";>File Submission</p>
</div>

#  Merging the predictions of the 2 models and file submission <a id='Merging'></a>
[Go back to the agenda](#Agenda)

In the code below, I merged the output of the 2 models so the output of the specific model overwrites the output of the generic model for the passengers traveling in group.

In [16]:
# output_full = output of the generic model 
output_full = pd.DataFrame({'PassengerId': test_full['PassengerId'], 'Survived': y_test_full.astype('int')})

# output_alone = subset of output_full. Only passengers travelling alone or having all their group members on the test set are retained in this output. 
output_alone = output_full[output_full['PassengerId'].isin(test_full[(test_full['GroupSize']==1) | (test_full['NbSurvivors']+test_full['NbDead']==0)]['PassengerId'].to_list())].copy()

# output_full = output of the specific model. This model was applied to passengers traveling in group and having one or more group members in train set 
output_group = pd.DataFrame({'PassengerId': test_group['PassengerId'], 'Survived': y_test_group.astype('int')})

# concatenating the 2 outputs and submitting the file
output = pd.concat([output_alone, output_group]).sort_values(by='PassengerId')
output.to_csv('my_submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
